# Precompute Icon Bubble Colors

This notebook scans the cuisine icon set, computes alpha-weighted average colors, blends each color toward white, and exports a TypeScript color map for marker bubbles.

In [26]:
from pathlib import Path
import json

try:
    from PIL import Image
except ImportError as exc:
    raise ImportError('Pillow is required. Install with: pip install pillow') from exc

WHITE_BLEND_FACTOR = 0.5
ALPHA_THRESHOLD = 0.05
MAX_SAMPLE_SIZE = 32

def find_workspace_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'package.json').exists() and (parent / 'src').exists():
            return parent
    raise FileNotFoundError('Could not locate workspace root from current directory.')

workspace_root = find_workspace_root(Path.cwd().resolve())
icon_dir = workspace_root / 'src' / 'assets' / 'icon_cuisines'
out_file = workspace_root / 'src' / 'MapPage' / 'components' / 'Map' / 'DataLayer' / 'TopPlacesLayer' / 'syncMarkers' / 'markers' / 'backdropColors' / 'preDerivedIconColors.ts'

if not icon_dir.exists():
    raise FileNotFoundError(f'Icon directory not found: {icon_dir}')

print('Workspace root:', workspace_root)
print('Icon dir:', icon_dir)
print('Output file:', out_file)

Workspace root: C:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer
Icon dir: C:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\src\assets\icon_cuisines
Output file: C:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\src\MapPage\components\Map\DataLayer\TopPlacesLayer\syncMarkers\markers\backdropColors\preDerivedIconColors.ts


In [27]:
def blend_channel_with_white(value: int, factor: float) -> int:
    return round((value * (1 - factor)) + (255 * factor))

def average_icon_rgba(path: Path, max_sample_size: int = 32, alpha_threshold: float = 0.05):
    with Image.open(path).convert('RGBA') as img:
        w, h = img.size
        max_dim = max(w, h)
        if max_dim <= 0:
            return None

        scale = min(1.0, max_sample_size / max_dim)
        sample_w = max(1, round(w * scale))
        sample_h = max(1, round(h * scale))
        img = img.resize((sample_w, sample_h), Image.Resampling.LANCZOS)

        pixels = img.getdata()
        weighted_r = 0.0
        weighted_g = 0.0
        weighted_b = 0.0
        total_weight = 0.0

        for r, g, b, a in pixels:
            alpha = a / 255.0
            if alpha <= alpha_threshold:
                continue

            weighted_r += r * alpha
            weighted_g += g * alpha
            weighted_b += b * alpha
            total_weight += alpha

        if total_weight == 0:
            return None

        avg_r = round(weighted_r / total_weight)
        avg_g = round(weighted_g / total_weight)
        avg_b = round(weighted_b / total_weight)
        return avg_r, avg_g, avg_b

def tinted_rgb(rgb, white_factor: float = 0.64):
    if rgb is None:
        return None
    r, g, b = rgb
    return (
        blend_channel_with_white(r, white_factor),
        blend_channel_with_white(g, white_factor),
        blend_channel_with_white(b, white_factor),
    )

def rgb_to_css(rgb):
    r, g, b = rgb
    return f'rgb({r}, {g}, {b})'

In [28]:
icon_paths = sorted(icon_dir.glob('*.png'))
if not icon_paths:
    raise RuntimeError(f'No PNG icons found in {icon_dir}')

precomputed_map = {}
rows = []

for icon_path in icon_paths:
    key = icon_path.stem.lower()
    avg = average_icon_rgba(icon_path, max_sample_size=MAX_SAMPLE_SIZE, alpha_threshold=ALPHA_THRESHOLD)
    tinted = tinted_rgb(avg, white_factor=WHITE_BLEND_FACTOR)

    if tinted is None:
        continue

    css_color = rgb_to_css(tinted)
    precomputed_map[key] = css_color
    rows.append((key, avg, tinted, css_color))

print(f'Computed colors for {len(precomputed_map)} icons.')
rows[:10]

Computed colors for 40 icons.


C:\Users\tichen\AppData\Local\Temp\ipykernel_35724\3535853204.py:16: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = img.getdata()


[('african', (166, 117, 85), (210, 186, 170), 'rgb(210, 186, 170)'),
 ('american', (252, 159, 22), (254, 207, 138), 'rgb(254, 207, 138)'),
 ('asian', (255, 232, 157), (255, 244, 206), 'rgb(255, 244, 206)'),
 ('australian', (242, 129, 9), (248, 192, 132), 'rgb(248, 192, 132)'),
 ('bakery', (253, 179, 74), (254, 217, 164), 'rgb(254, 217, 164)'),
 ('bistro', (106, 179, 240), (180, 217, 248), 'rgb(180, 217, 248)'),
 ('breakfast', (231, 216, 191), (243, 236, 223), 'rgb(243, 236, 223)'),
 ('british', (244, 211, 113), (250, 233, 184), 'rgb(250, 233, 184)'),
 ('buffet', (126, 178, 217), (190, 216, 236), 'rgb(190, 216, 236)'),
 ('burgers', (191, 151, 35), (223, 203, 145), 'rgb(223, 203, 145)')]

In [29]:
def build_ts_module(color_map: dict[str, str], white_factor: float) -> str:
    sorted_items = sorted(color_map.items(), key=lambda kv: kv[0])

    lines = [
        '// Auto-generated by precompute_icon_colors.ipynb',
        f'// White blend factor: {white_factor}',
        'const preDerivedIconColors: Record<string, string> = {',
    ]

    for key, value in sorted_items:
        lines.append(f"  '{key}': '{value}',")

    lines.extend([
        '};',
        '',
        'export default preDerivedIconColors;',
    ])

    return '\n'.join(lines) + '\n'

ts_content = build_ts_module(precomputed_map, WHITE_BLEND_FACTOR)
out_file.write_text(ts_content, encoding='utf-8')

print(f'Wrote TypeScript color map to: {out_file}')

Wrote TypeScript color map to: C:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\src\MapPage\components\Map\DataLayer\TopPlacesLayer\syncMarkers\markers\backdropColors\preDerivedIconColors.ts


In [30]:
print(json.dumps(precomputed_map, indent=2, ensure_ascii=True)[:3000])

{
  "african": "rgb(210, 186, 170)",
  "american": "rgb(254, 207, 138)",
  "asian": "rgb(255, 244, 206)",
  "australian": "rgb(248, 192, 132)",
  "bakery": "rgb(254, 217, 164)",
  "bistro": "rgb(180, 217, 248)",
  "breakfast": "rgb(243, 236, 223)",
  "british": "rgb(250, 233, 184)",
  "buffet": "rgb(190, 216, 236)",
  "burgers": "rgb(223, 203, 145)",
  "cafe": "rgb(248, 174, 150)",
  "chinese": "rgb(176, 210, 189)",
  "deli": "rgb(240, 171, 153)",
  "desert": "rgb(247, 196, 190)",
  "east european": "rgb(193, 180, 169)",
  "european": "rgb(180, 217, 248)",
  "family restaurant": "rgb(180, 217, 248)",
  "fast food": "rgb(250, 194, 151)",
  "fine dining": "rgb(248, 222, 222)",
  "french": "rgb(255, 220, 137)",
  "german": "rgb(252, 204, 135)",
  "halal": "rgb(240, 176, 178)",
  "italian": "rgb(247, 202, 144)",
  "japanese": "rgb(203, 190, 177)",
  "kebab": "rgb(240, 176, 178)",
  "korean": "rgb(253, 234, 227)",
  "latin american": "rgb(254, 215, 132)",
  "mediterranean": "rgb(168, 201, 1